In [6]:
import os
import numpy as np
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

# Імпортуємо ВАШУ функцію
from fd_functions import box_counting

In [7]:
DATA_PATH = "F:/datasets/Chest X-Ray"
TRAIN_DIR = os.path.join(DATA_PATH, "train")
TEST_DIR = os.path.join(DATA_PATH, "test")
TRAIN_LIST = os.path.join(DATA_PATH, "train.txt")
TEST_LIST = os.path.join(DATA_PATH, "test.txt")

IMG_SIZE = (512, 512)

In [8]:
preprocess_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])

In [9]:
def read_original_list(list_path):
    rows = []
    with open(list_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            _, fname, label_str, _src = parts
            rows.append({"filename": fname, "label_str": label_str})
    return pd.DataFrame(rows)

In [10]:
# =======================
# Головний цикл обробки
# =======================
def process_dataset(df, img_dir, desc="Processing"):
    fd_raw_list = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        img_path = os.path.join(img_dir, row["filename"])

        try:
            img = Image.open(img_path)
            if img.mode != "RGB":
                img = img.convert("RGB")

            img_t = preprocess_transform(img)

            # Реалізація АДАПТИВНОЇ бінаризації (по середньому значенню)
            # Рахуємо середнє значення яскравості тензора
            adaptive_thr = img_t.mean().item()

            # Викликаємо вашу функцію
            fd_val = box_counting(img_t, threshold=adaptive_thr)

            fd_raw_list.append(fd_val)
        except Exception as e:
            print(f"Помилка читання {img_path}: {e}")
            fd_raw_list.append(np.nan)

    df["fd_raw"] = fd_raw_list
    return df

In [11]:
print("1. Читання оригінальних списків...")
df_train = read_original_list(TRAIN_LIST)
df_test = read_original_list(TEST_LIST)

print(f"\n2. Обчислення ФР для Train датасету ({len(df_train)} зображень)...")
df_train = process_dataset(df_train, TRAIN_DIR, desc="Train FD")

print(f"\n3. Обчислення ФР для Test датасету ({len(df_test)} зображень)...")
df_test = process_dataset(df_test, TEST_DIR, desc="Test FD")

df_train = df_train.dropna()
df_test = df_test.dropna()

print("\n4. Нормалізація значень (Z-score normalization)...")
fd_mean = df_train["fd_raw"].mean()
fd_std = df_train["fd_raw"].std()

df_train["fd_norm"] = (df_train["fd_raw"] - fd_mean) / fd_std
df_test["fd_norm"] = (df_test["fd_raw"] - fd_mean) / fd_std

print(f"Статистика Train FD -> Mean: {fd_mean:.4f}, Std: {fd_std:.4f}")

train_out_path = os.path.join(DATA_PATH, "train_with_fd.csv")
test_out_path = os.path.join(DATA_PATH, "test_with_fd.csv")

df_train.to_csv(train_out_path, index=False)
df_test.to_csv(test_out_path, index=False)

print(f"\nГотово! Файли збережено:")
print(f"- {train_out_path}")
print(f"- {test_out_path}")

1. Читання оригінальних списків...

2. Обчислення ФР для Train датасету (29986 зображень)...


Train FD: 100%|██████████| 29986/29986 [08:32<00:00, 58.46it/s] 



3. Обчислення ФР для Test датасету (400 зображень)...


Test FD: 100%|██████████| 400/400 [00:20<00:00, 19.66it/s]



4. Нормалізація значень (Z-score normalization)...
Статистика Train FD -> Mean: 1.8601, Std: 0.0411

Готово! Файли збережено:
- F:/datasets/Chest X-Ray\train_with_fd.csv
- F:/datasets/Chest X-Ray\test_with_fd.csv


In [13]:
file_path = os.path.join(DATA_PATH, "train_with_fd.csv")

# Load the CSV file into a pandas DataFrame
created_dataset = pd.read_csv(file_path)

In [14]:
print("Dataset structure (first 5 rows):")
print(created_dataset.head())

Dataset structure (first 5 rows):
                                         filename label_str    fd_raw  \
0                                  ARDSSevere.png  negative  1.856335   
1  acute-respiratory-distress-syndrome-ards-1.jpg  negative  1.826723   
2    acute-respiratory-distress-syndrome-ards.jpg  negative  1.849207   
3          ards-secondary-to-tiger-snake-bite.png  negative  1.819336   
4                 pneumocystis-pneumonia-2-PA.png  negative  1.879612   

    fd_norm  
0 -0.091792  
1 -0.812817  
2 -0.265352  
3 -0.992692  
4  0.474981  


In [15]:
print("\nDataset info:")
print(created_dataset.info())


Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29986 entries, 0 to 29985
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   filename   29986 non-null  object 
 1   label_str  29986 non-null  object 
 2   fd_raw     29986 non-null  float64
 3   fd_norm    29986 non-null  float64
dtypes: float64(2), object(2)
memory usage: 937.2+ KB
None
